# Tutorial - Simple
A simple optimization problem related to production optimization under resource constraints.

dd

- - - 
IMPORT PACKAGE, DEFINE MODEL NAME AND ROOT PATH

In [ ]:
import cvxlab

- - - 
MODEL DIRECTORY GENERATION

Generation of a model directory in a defined path. Notes:
- Blank .yml or .xlsx files can be generated
- API_reference.ipynb can be generated as guidance for main APIs
- user_operators.py and user_constants.py templates can be generated

In [ ]:
cvxlab.create_model_dir(
    settings_file_type='xlsx', 
    include_user_defined_templates=True,
    Uncertainty=True
)

- - - 
MODEL GENERATION FROM SCRATCH

Step-by-step model creation with sets, data and problems generation.

- Generation of a Model instance defined by settings files filled by user
- model_settings_from: select among yml and xlsx
- use_existing_data=True : model coordinates are loaded and numerical problem initialized (model ready to be run). 
- use_existing_data=False : sets excel file generated only, to be filled by user

In [ ]:
model = cvxlab.Model(
    log_level='debug',
    use_existing_data=False,
    detailed_validation=True,
    Uncertainty=True
)

Once sets.xlsx file has filled with sets data:

- load_model_coordinates(): load user defined sets, fill coordinates
- initialize_blank_data_structure():
    - generate blank sqlite database with sets and variables (empty numerical values)
    - generate blank xlsx files for exogenous data to be filled by the user

In [ ]:
model.initialize_model_environment()

Once exogenous xlsx files has filled:

- load_exogenous_data_to_sqlite_database(): load input data into sqlite database
- initialize_problems(): load and validate symbolic problem, generate numerical problem 

In [ ]:
model.refresh_database_and_initialize_problem()

- - - 
MODEL GENERATION FROM EXISTING DATA

If use_existing_data=True, generation of Model instance working with existing database and data input files.

The model is generated and ready to be solved.

In [ ]:
model = cvxlab.Model(
    log_level='debug',
    model_settings_from='xlsx',
    use_existing_data=True,
    detailed_validation=True,
)

In [ ]:
model.run_model(
    verbose=True,
    # solver='GUROBI',
    integrated_problems=True,
)

- - - 
SAVING/LOADING MODEL INSTANCE

In case model instance generation is taking huge time, it is possible to save/load model instance to avoid regenerating it several times.

Notice that - for unknown reasons - model instance can be loaded only if it has not already solve (i.e. do not save model instance if you run the model!)

In [ ]:
cvxlab.handle_model_instance(
    action='save',
    instance=model,
    file_name='instance_name'
)

In [ ]:
model = cvxlab.handle_model_instance(
    action='load',
    file_name='file_name',
    source_dir_path=r"D:\git_repos\pyesm\tests\models\integrated\1_coupled_model\instances"
)

- - -
DATA and SYMBOLIC PROBLEM UPDATE without regenerating Model instance

- initialize_problems(): if symbolic problem has modified, upload it and generates a new numerical model

- update_database_and_problem(): in case of modifications in input data files (but not in sets, nor in variables structures) and symbolic problem, update database and problem

- reinitialize_sqlite_database(): reinitialize sqlite database endogenous variables


In [ ]:
# in case of need for generating only one or more data input files/tabs
model.generate_input_data_files(table_key_list=[])

In [ ]:
# in case of modifications in symbolic problem only,
# update problems dataframe and symbolic problem
model._initialize_problems()

In [ ]:
# in case of modifications in input data files (but not in sets, nor in 
# variables structures) and symbolic problem, update database and problem
model.refresh_database_and_initialize_problem()

In [ ]:
# in case of modifications in input data files (but not in sets, nor in 
# variables structures) and symbolic problem, reinitialize sqlite database for
# endogenous variables
model.reinitialize_sqlite_database(force_overwrite=True)

- - -
SOLVE NUMERICAL PROBLEM and ENDOGENOUS DATA EXPORT

- run_model(): run model with various settings
- load_results_to_database(): once model has successfully solved, load endogenous parameters data to sqlite database.
- check_model_results(): check model results compared to an existing database (path and name of the database with the expected results in model settings attributes)

In [ ]:
model.run_model(
    verbose=False,
    solver='GUROBI',
    integrated_problems=True,
)

In [ ]:
model.load_results_to_database(force_overwrite=True)

In [ ]:
model.check_model_results()

- - -
UTILITIES

- sets(): list of model sets
- variables(): dictionary with variable name (key) and shape of teh variable (value)
- variable(name, ...): visual check of variables content in the database
- set(name): visual check of sets content in the database

In [ ]:
model.variables
model.sets

In [ ]:
model.set('products')

In [ ]:
model.variable(name='L',)